## 便于自主编程的gjf文件生成工具——gjftools(v0.1.3)

### 0. 安装

1. 可以将`gjftools.py`文件复制到需要使用的文件夹中直接调用（不推荐）

2. 首先进入某个虚拟环境（基础环境也可，但同样不推荐）；然后进入`gjftools/`文件夹，运行`pip install .`命令，看到最后输出`Successfully installed gjftools-0.1.2`即为安装成功，之后只要在该虚拟环境下，无论在什么位置都可以直接`from gjftools import gjfdata`。

In [4]:
import os
import sys
from gjftools import gjfdata

### 1. 基本使用方法
1.1. 初始化一个gjfdata对象（可以认为是一个gjf文件）  
1.2. 指定结构信息来源  
1.3. 指定gjf文件模板（gjf文件头尾信息来源）  
1.4. 根据实际需求调整/添加额外信息  
1.5. 输出gjf文件

In [2]:
test_logfile = 'example/structures/tsmod-opt.log'
test_modelfile = 'example/models/ts.gjf'
test_outputfile = 'example/results/tsmod-opt-ts-simple.gjf'

# 极简版
gjf1 = gjfdata()  # 初始化gjfdata对象
gjf1.get_body_from_log(test_logfile)  # 指定log文件路径
gjf1.get_model_file(test_modelfile)  # 指定gjf文件模板
gjf1.write_gjf(test_outputfile)  # 输出gjf文件

In [3]:
test_logfile = 'example/structures/tsmod-opt.log'
test_modelfile = 'example/models/ts.gjf'
test_outputfile = 'example/results/tsmod-opt-ts-full.gjf'

# 完整版
gjf1 = gjfdata(verbose=True)  # 初始化gjfdata对象(指定详细输出)

gjf1.get_body_from_log(logfile=test_logfile,  # 指定log文件路径（必填）
                       structure_index=-1)  # 指定使用的log文件里的结构编号，从0开始（选填，默认-1）

gjf1.get_model_file(modelfile=test_modelfile)  # 指定gjf文件模板（必填）

gjf1.nproc = 4  # 指定使用核数

gjf1.mem = '4GB'  # 指定使用内存

gjf1.chk_name = 'Xu6-test-check.chk'  # 指定chk文件名
# chk可加可不加，非必填，默认与output_name一致

gjf1.input_para = '# opt freq genecp scrf=(smd,solvent=generic,read) nosymm em=gd3bj pbe1pbe'

gjf1.title = 'Xu6-test-title'  # 指定运算任务标题
# 非必填，默认与output_name一致

gjf1.c_m = [0,2]  # 指定电荷及多重度，非必填，如body文件中有相关信息将自动读取，若无则默认0 1

gjf1.atom_info[6] = '(Iso=2)'  # 指定特定原子的额外信息（目前只有同位素指定中涉及到）

gjf1.output_name = test_outputfile  # 指定输出文件名
# gjf可加可不加，非必填，默认bodyfilename_modelfilename
# 若此处给出名称不包括路径，则默认输出在bodyfile路径

gjf1.write_gjf(output_name=test_outputfile,  # 指定输出文件名，非必填，会覆盖上面output_name
               need_chk=True,  # 指定是否需要添加chk命令，非必填，默认为True
               check_basis=True,  # 指定是否要对basis部分进行元素检查（偶尔在复杂情况下会出现错误），非必填，默认为True
               )

read structure info from example/structures/tsmod-opt.log structure -1
89 structures in example/structures/tsmod-opt.log.
read head and tail from example/models/ts.gjf.
write gjf file to example/results/tsmod-opt-ts-full.gjf.


#### 1. 从log文件生成gjf文件

**方法a：直接写代码**

实例1：tsmod运算完成后进行ts计算

In [5]:
log_file = '/home/yzh/gjftools/example/structures/tsmod-opt.log'
model_file = '/home/yzh/gjftools/example/models/ts.gjf'
result_file = '/home/yzh/gjftools/example/results/tsmod-opt-ts-direct.gjf'

gjf = gjfdata(verbose=1)
gjf.get_body_from_log(log_file)
gjf.get_model_file(model_file)
gjf.write_gjf(result_file, need_chk=False)

read structure info from /home/yzh/gjftools/example/structures/tsmod-opt.log structure -1
89 structures in /home/yzh/gjftools/example/structures/tsmod-opt.log.
read head and tail from /home/yzh/gjftools/example/models/ts.gjf.
write gjf file to /home/yzh/gjftools/example/results/tsmod-opt-ts-direct.gjf.


**方法b：将代码封装为函数**  

好处：便于重复使用和批量操作  
实例2：VCD谱图计算

In [4]:
def log_to_gjf(logfile, model_file, result_file):
    gjf = gjfdata()
    gjf.get_body_from_log(logfile)
    gjf.get_model_file(model_file)
    gjf.atom_info[21] = '(Iso=2)'
    gjf.write_gjf(result_file)

In [5]:
model_file = 'example/models/VCD.gjf'
source_dir = 'example/structures/vcdoptlog'
result_dir = 'example/results/vcdgjf'

loglist = [os.path.join(source_dir, f)
           for f in os.listdir(source_dir) if f.endswith('.log')]

for logfile in loglist:
    result_file = os.path.join(result_dir, os.path.basename(logfile))
    log_to_gjf(logfile, model_file, result_file)

实例3：从irc文件中选取ts结构创建新gjf文件

In [6]:
irc_file = 'example/structures/IRC.log'
model_file = 'example/models/ts.gjf'

irc_index = 10
result_file = f'example/results/IRC-{irc_index+1}-direct.gjf'

gjf = gjfdata(verbose=1)
gjf.get_body_from_irc(irc_file, irc_index=10)
gjf.get_model_file(model_file)
gjf.write_gjf(result_file)

read structure info from example/structures/IRC.log irc_index 10
22 structures in example/structures/IRC.log.
read head and tail from example/models/ts.gjf.
write gjf file to example/results/IRC-11-direct.gjf.


实例4：从log文件震荡位置出发创建新gjf文件（练习）

假设从第50个点开始认为震荡

In [3]:
log_file = 'example/structures/tsmod-opt.log'
model_file = 'example/models/tsmod-opt.gjf'
result_file = 'example/results/tsmod-opt-1.gjf'

gjf = gjfdata()
gjf.get_body_from_log(log_file, structure_index=49)
gjf.get_model_file(model_file)
gjf.write_gjf(result_file)

实例5：从irc文件中各点提取结构并分别创建新gjf文件（练习）

共有21个点，输出文件命名中需要包含点编号

In [4]:
def irc_to_gjf(irc_file, model_file, irc_index, result_file):
    gjf = gjfdata()
    gjf.get_body_from_irc(irc_file, irc_index)
    gjf.get_model_file(model_file)
    gjf.write_gjf(result_file)

In [5]:
irc_file = 'example/structures/IRC.log'
model_file = 'example/models/sp.gjf'

for i in range(21):
    result_file = f'example/results/IRC-{i+1}-function.gjf'
    irc_to_gjf(irc_file, model_file, i, result_file)

#### 2. 从xyz文件生成gjf文件

实例1：从单个结构xyz文件生成gjf文件

In [7]:
xyz_file = 'example/structures/single-structure.xyz'
model_file = 'example/models/tsmod-opt.gjf'
result_file = 'example/results/single-xyz-tsmod-opt.gjf'

gjf = gjfdata()
gjf.get_body_from_xyz(xyz_file)
gjf.get_model_file(model_file)
gjf.write_gjf(result_file, need_chk=True)

实例2：从rpipmin输出的xyz文件（构象搜索结果）批量生成gjf文件

In [8]:
def rpipmin_to_gjf(xyz_file, tstmpe, model_file, result_file):
    gjf = gjfdata()
    gjf.get_body_from_rpipmin(xyz_file, tstmpe_index=tstmpe)
    gjf.get_model_file(model_file)
    gjf.write_gjf(result_file, need_chk=True)

In [9]:
xyz_file = 'example/structures/rpipmin-cf-result.xyz'
model_file = 'example/models/basic-opt.gjf'
tstmpe_list = [1, 404, 9, 1869, 8, 2]

for tstmpe in tstmpe_list:
    result_file = f'example/results/rpipmin-cf-{str(tstmpe)}-opt.gjf'
    rpipmin_to_gjf(xyz_file, tstmpe, model_file, result_file)

实例3：从crest输出的xyz文件批量生成gjf文件（练习）

In [10]:
def crest_to_gjf(xyz_file, index, model_file, result_file):
    gjf = gjfdata()
    gjf.get_body_from_xyz(xyz_file, structure_index=index)
    gjf.get_model_file(model_file)
    gjf.write_gjf(result_file, need_chk=True)

In [13]:
xyz_file = '/home/yzh/gjftools/example/structures/crest_cf_result.xyz'
model_file = '/home/yzh/gjftools/example/models/basic-opt.gjf'
structure_index_list = [0, 3, 4]

for i in structure_index_list:
    result_file = f'/home/yzh/gjftools/example/results/crest_cf_result-{i+1:0=3d}-opt.gjf'
    crest_to_gjf(xyz_file, i, model_file, result_file)

#### 3. 从gjf文件生成gjf文件

In [5]:
def gjf_to_gjf(gjf_file, model_file, result_file):
    gjf = gjfdata()
    gjf.get_body_from_gjf(gjf_file)
    gjf.get_model_file(model_file)
    gjf.write_gjf(result_file)

In [6]:
gjf_file = 'example/models/ts.gjf'
model_file = 'example/models/sp.gjf'
result_file = 'example/results/ts-sp.gjf'

gjf_to_gjf(gjf_file, model_file, result_file)

#### 4. 从sdf文件生成gjf文件

In [7]:
sdf_file = 'example/structures/multiple-confs.sdf'
model_file = 'example/models/basic-opt.gjf'
result_file = 'example/results/sdf-test.gjf'

gjf = gjfdata()
gjf.get_body_from_sdf(sdf_file, read_title=False)
gjf.get_model_file(model_file)
gjf.write_gjf(result_file)

#### 5. 从modredundant scan log文件中生成gjf文件

In [8]:
mod_file = 'example/structures/xtbscan.log'
gaumod_file = 'example/structures/gauscan.log'
model_file = 'example/models/basic-opt.gjf'
opt_result_file = 'example/results/xtbscan-opt.gjf'
ts_result_file = 'example/results/xtbscan-ts.gjf'
gau_result_file = 'example/results/gauscan-ts.gjf'

gjf = gjfdata(verbose=1)
gjf.get_body_from_xtbscan(mod_file)  # last structure
gjf.get_model_file(model_file)
gjf.write_gjf(opt_result_file)

gjf.get_body_from_xtbscan(mod_file, highest_energy=True)  # get quasi-ts structure
gjf.write_gjf(ts_result_file)

gjf.get_body_from_gauscan(gaumod_file, highest_energy=True)
gjf.write_gjf(gau_result_file)

read structure info from example/structures/xtbscan.log scan point -1
16 structures in example/structures/xtbscan.log.
read head and tail from example/models/basic-opt.gjf.
write gjf file to example/results/xtbscan-opt.gjf.
read structure info from example/structures/xtbscan.log scan point 6
16 structures in example/structures/xtbscan.log.
write gjf file to example/results/xtbscan-ts.gjf.
read structure info from example/structures/gauscan.log scan point 8
11 structures in example/structures/gauscan.log.
write gjf file to example/results/gauscan-ts.gjf.
